# MFI: Modern Fortran Interfaces — GPU/CuBLAS Test

This notebook builds and tests MFI with GPU-accelerated cuBLAS on a Colab Tesla T4.

## Requirements
- GPU runtime (T4, V100, A100, etc.)
- CUDA toolkit installed (Colab includes it by default)


In [ ]:
#@title 1. Environment check
!nvidia-smi
!nvcc --version

In [ ]:
#@title 2. Install build dependencies (gfortran, fypp, fpm, BLAS/LAPACK)
!apt-get update -qq
!apt-get install -qq -y gfortran libblas-dev liblapack-dev python3-pip > /dev/null 2>&1
!pip install fypp -q

# Download fpm binary (Fortran Package Manager)
!wget -q -O fpm https://github.com/fortran-lang/fpm/releases/download/v0.10.1/fpm-0.10.1-linux-x86_64
!chmod +x fpm
!mv fpm /usr/local/bin/

print("\n✅ Dependencies installed")
!gfortran --version | head -1
!fpm --version
!fypp --version

In [ ]:
#@title 3. Clone MFI repository
import os
os.chdir("/content")
!git clone https://github.com/14NGiestas/mfi.git
os.chdir("/content/mfi")

print("\n✅ Cloned MFI")
!ls -la

In [ ]:
#@title 4. Preprocess and build with cuBLAS
import os
os.chdir("/content/mfi")

# Set CUDA paths
cuda_include = "/usr/local/cuda/include"
cuda_lib = "/usr/local/cuda/lib64"
os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"

# Preprocess
!make clean && make

# Build with fpm, passing CUDA include path
!fpm build --flag "-I{cuda_include}"

print("\n✅ Build complete")

In [ ]:
#@title 5. Run tests with GPU enabled
import os
os.chdir("/content/mfi")
os.environ["MFI_USE_CUBLAS"] = "1"
os.environ["LD_LIBRARY_PATH"] = f"/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

!fpm test 2>&1

In [ ]:
#@title 6. Quick GEMV GPU smoke test
%%writefile /content/mfi/test/gpu_smoke.f90
program gpu_smoke
    use iso_fortran_env
    use mfi_blas, only: mfi_execution_init, mfi_sgemv, mfi_get_execution_mode
    implicit none
    real(real32) :: M(4,4), X(4), Y(4), Y_ref(4)
    real(real32) :: alpha, beta
    integer :: i, j

    call mfi_execution_init()
    print *, "Execution mode:", trim(mfi_get_execution_mode())

    call random_number(M)
    call random_number(X)
    Y = 0.0
    Y_ref = 0.0
    alpha = 2.0
    beta = 0.5

    ! CPU reference: Y = alpha*M*X + beta*Y
    do i = 1, 4
        do j = 1, 4
            Y_ref(i) = Y_ref(i) + alpha * M(i,j) * X(j)
        end do
        Y_ref(i) = Y_ref(i) + beta * Y(i)
    end do

    ! GPU computation
    Y = 0.0
    call mfi_sgemv(M, X, Y, alpha=alpha, beta=beta, trans='N')

    print *, "CPU ref Y:", Y_ref
    print *, "GPU    Y:", Y
    print *, "Diff   :", abs(Y - Y_ref)

    if (all(abs(Y - Y_ref) < 1.0e-4)) then
        print *, "✅ GEMV GPU PASSED"
    else
        print *, "❌ GEMV GPU FAILED"
    end if
end program


In [ ]:
#@title 7. Compile and run smoke test
import os
os.chdir("/content/mfi")
os.environ["MFI_USE_CUBLAS"] = "1"
os.environ["LD_LIBRARY_PATH"] = f"/usr/local/cuda/lib64:/usr/local/cuda/lib64/stubs:{os.environ.get('LD_LIBRARY_PATH', '')}"

!gfortran -DMFI_EXTENSIONS -DMFI_USE_CUBLAS -I/usr/local/cuda/include \
    -I./src/mfi -I./src/f77 -I. \
    test/gpu_smoke.f90 src/f77/blas.f90 src/mfi/blas.f90 \
    -lblas -llapack -lcublas -lcudart \
    -L/usr/local/cuda/lib64 -o gpu_smoke 2>&1

!echo "\n=== Running GPU smoke test ==="
!LD_LIBRARY_PATH=/usr/local/cuda/lib64:/usr/local/cuda/lib64/stubs:$LD_LIBRARY_PATH ./gpu_smoke